# Week 9: Assert-Based Testing
### PHASE 5: Output & Verification

*Core Mastery: "I can write systematic tests to verify each stage of my pipeline"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. Explain why testing is essential for reliable software
2. Write `assert` statements with custom error messages
3. Test functions with expected normal-case inputs
4. Design tests for edge cases (empty inputs, zeros, negatives)
5. Combine `try/except` with `assert` to test error handling
6. Compare floating-point results safely with epsilon tolerance
7. Test individual stages of a multi-function pipeline
8. Build a reusable `run_tests()` function with pass/fail summary
9. Understand the test-first mindset and its benefits
10. Apply systematic testing to NumPy-based computations

## 🎯 Core Mastery Connection

You have learned to build data-processing functions (Weeks 4-6), process arrays with NumPy (Week 7), and visualize results (Week 8). But how do you **know** your code is correct? Testing is the answer. The `assert` statement is Python's simplest testing tool — and this week you will use it to verify every stage of a pipeline.

---
## 🤝 Mechatronics Learning Contract

- **Professional relevance:** examples and core exercises model the data, sensing, automation, numerical, and decision tasks used in mechatronics engineering.
- **Interaction:** predict before running, compare reasoning with a partner, and ask whenever a step is unclear; scheduled checkpoints guarantee question time.
- **Assessment alignment:** worked examples and Core Exercises 1–8 rehearse the same reasoning operations used on exams—trace, implement, debug, interpret, and justify—while exam values and contexts may change.
- **Learning evidence:** weekly notebooks remain private practice. Non-exam evidence comes from scheduled in-class project demonstrations/presentations using a published rubric, not homework collection.


---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Checkpoints provide immediate feedback only inside your Colab runtime. Nothing
is transmitted, saved for grading, or reviewed by the instructor. Exercises 9
and above are optional extensions—not homework.


In [ ]:
# Run this setup cell once at the start of class.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = (int(correct), 1)
    if correct:
        print(f"✅ Checkpoint {number}: correct")
        print("Why:", explanation)
    elif not str(answer).strip():
        print(f"🟡 Checkpoint {number}: enter an answer, then run this cell again.")
    else:
        print(f"🔴 Checkpoint {number}: not yet. Review the preceding examples and retry.")
    return correct

def exercise_checkpoint(number, expected=8):
    """Count core exercise cells that contain work and were run in this runtime."""
    import re
    completed = set()
    for source in globals().get("In", []):
        match = re.search(r"#\s*✏️\s*\[EX(\d+)\]", str(source), flags=re.I)
        if not match:
            continue
        answer = re.sub(r"^.*?#\s*✏️\s*\[EX\d+\]", "", str(source), count=1, flags=re.I | re.S).strip()
        if answer and answer != "pass" and "your code here" not in answer.lower():
            completed.add(int(match.group(1)))
    checks = [(index, index in completed) for index in range(1, expected + 1)]
    passed = sum(done for _, done in checks)
    _checkpoint_results[int(number)] = (passed, expected)
    print(f"Core Practice: {passed}/{expected} exercise cells edited and run")
    missing = [str(index) for index, done in checks if not done]
    if not missing:
        print("✅ Core Practice complete.")
    else:
        print("🟡 Still to complete/run:", ", ".join(missing))
    return passed, expected

def show_progress_summary():
    print("\n=== My local progress ===")
    for number in range(1, 6):
        if number in _checkpoint_results:
            passed, total = _checkpoint_results[number]
            print(f"Checkpoint {number}: {passed}/{total}")
        else:
            print(f"Checkpoint {number}: not run")
    print("Results exist only in this temporary runtime.")

print("✅ Local self-check tools ready")


---
## Part 1: Why Test? Bugs Are Expensive

| When bug found | Cost to fix |
|---|---|
| While coding | Minutes |
| During testing | Hours |
| In production | Days to weeks |
| After release | Reputation + money |

Testing is not optional — it is how professional engineers ship reliable software. We start with the simplest tool: **assert**.

In [ ]:
# A simple function with a bug
def hesapla_ortalama(notlar):
    """Calculate average of a list of grades."""
    return sum(notlar) / len(notlar)

# Does it work?
print(hesapla_ortalama([80, 90, 70]))   # 80.0 — correct!
print(hesapla_ortalama([100]))           # 100.0 — correct!
# print(hesapla_ortalama([]))            # ZeroDivisionError! Bug!

**Figure 1.1** — A function that works for normal inputs but crashes on an empty list.

In [ ]:
# Fixed version
def hesapla_ortalama(notlar):
    """Calculate average of a list of grades."""
    if not notlar:
        return 0.0
    return sum(notlar) / len(notlar)

print(hesapla_ortalama([80, 90, 70]))  # 80.0
print(hesapla_ortalama([]))            # 0.0 — no crash!

**Figure 1.2** — The fixed function handles edge cases gracefully.

---
## Part 2: The assert Statement

Syntax:
```python
assert condition, "error message"
```

| Part | Meaning |
|---|---|
| `condition` | A boolean expression |
| Error message | Shown if condition is `False` |
| `AssertionError` | Raised when assert fails |

Assert means: *"I guarantee this is true. If it's not, stop immediately."*

In [ ]:
# Basic assert examples
x = 10
assert x > 0, "x must be positive"
print("Passed: x > 0")

assert isinstance(x, int), "x must be an integer"
print("Passed: x is int")

assert x != 0, "x must not be zero"
print("Passed: x is not zero")

print("\nAll assertions passed!")

**Figure 2.1** — Basic assert statements that all pass.

In [ ]:
# What happens when assert fails?
try:
    result = -5
    assert result >= 0, f"Expected non-negative, got {result}"
except AssertionError as e:
    print(f"AssertionError caught: {e}")

**Figure 2.2** — Catching an AssertionError to see the custom message.

In [ ]:
# Assert with function return values
def kare_al(n):
    return n ** 2

assert kare_al(3) == 9, "3 squared should be 9"
assert kare_al(0) == 0, "0 squared should be 0"
assert kare_al(-4) == 16, "-4 squared should be 16"
print("All kare_al tests passed!")

**Figure 2.3** — Using assert to test function return values.

---
### ⏱️ Checkpoint 1 of 5 — Assertions (target 00:55)

What happens when an assert condition is false?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, 'AssertionError',
    'A failed assertion raises `AssertionError`.',
)


---
## Part 3: Testing Normal Cases

Normal case testing verifies that the function works for **typical, expected inputs**.

Pattern:
```python
assert function(typical_input) == expected_output, "description"
```

In [ ]:
def toplam(liste):
    """Return sum of numbers in a list."""
    return sum(liste)

def ortalama(liste):
    """Return average of numbers in a list."""
    if not liste:
        return 0.0
    return sum(liste) / len(liste)

# Normal case tests
assert toplam([1, 2, 3]) == 6, "Sum of [1,2,3] should be 6"
assert toplam([10, 20]) == 30, "Sum of [10,20] should be 30"
assert ortalama([80, 90, 70]) == 80.0, "Average of [80,90,70] should be 80"
assert ortalama([100, 100]) == 100.0, "Average of [100,100] should be 100"

print("All normal case tests passed!")

**Figure 3.1** — Testing sum and average with typical inputs.

In [ ]:
def harf_notu(puan):
    """Convert numeric score to letter grade."""
    if puan >= 90: return 'A'
    if puan >= 80: return 'B'
    if puan >= 70: return 'C'
    if puan >= 60: return 'D'
    return 'F'

# Test each grade boundary
assert harf_notu(95) == 'A', "95 should be A"
assert harf_notu(85) == 'B', "85 should be B"
assert harf_notu(75) == 'C', "75 should be C"
assert harf_notu(65) == 'D', "65 should be D"
assert harf_notu(50) == 'F', "50 should be F"

# Boundary values
assert harf_notu(90) == 'A', "90 should be A"
assert harf_notu(89) == 'B', "89 should be B"
assert harf_notu(80) == 'B', "80 should be B"
assert harf_notu(79) == 'C', "79 should be C"

print("All letter grade tests passed!")

**Figure 3.2** — Testing boundary values for a letter-grade function.

---
## Part 4: Testing Edge Cases

Edge cases are unusual but valid inputs:

| Edge Case | Example |
|---|---|
| Empty input | `[]`, `""`, `0` |
| Single element | `[42]` |
| All same values | `[5, 5, 5, 5]` |
| Negative numbers | `[-3, -1, -7]` |
| Very large values | `[10**9]` |
| Zero | `0`, `[0, 0, 0]` |

In [ ]:
def bul_maksimum(liste):
    """Find maximum value in a list. Returns None for empty list."""
    if not liste:
        return None
    en_buyuk = liste[0]
    for x in liste[1:]:
        if x > en_buyuk:
            en_buyuk = x
    return en_buyuk

# Edge case tests
assert bul_maksimum([]) is None, "Empty list should return None"
assert bul_maksimum([42]) == 42, "Single element should return that element"
assert bul_maksimum([5, 5, 5]) == 5, "All same should return that value"
assert bul_maksimum([-3, -1, -7]) == -1, "Negative list: max is -1"
assert bul_maksimum([0, 0, 0]) == 0, "All zeros should return 0"
assert bul_maksimum([1, -1, 0]) == 1, "Mixed signs: max is 1"

print("All edge case tests passed!")

**Figure 4.1** — Testing a maximum-finding function with edge cases.

In [ ]:
def ters_cevir(metin):
    """Reverse a string."""
    return metin[::-1]

# Edge cases for strings
assert ters_cevir("") == "", "Empty string reversed is empty"
assert ters_cevir("a") == "a", "Single char stays same"
assert ters_cevir("aba") == "aba", "Palindrome stays same"
assert ters_cevir("merhaba") == "abahrem", "Normal reversal"
assert ters_cevir("12345") == "54321", "Digits reversed"

print("All string edge case tests passed!")

**Figure 4.2** — Testing string reversal with edge cases.

In [ ]:
def guvenlibolen(a, b):
    """Safe division that returns None for division by zero."""
    if b == 0:
        return None
    return a / b

assert guvenlibolen(10, 2) == 5.0
assert guvenlibolen(0, 5) == 0.0
assert guvenlibolen(10, 0) is None
assert guvenlibolen(-6, 3) == -2.0
assert guvenlibolen(7, 2) == 3.5

print("All safe division tests passed!")

**Figure 4.3** — Testing safe division including division by zero.

---
### ⏱️ Checkpoint 2 of 5 — Coverage (target 01:55)

Should tests include normal, boundary, and invalid cases? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'yes',
    'Different classes of input expose different bugs.',
)


---
## Part 5: Testing Error Handling

Sometimes we want to verify that a function **raises** an error for bad input. Pattern:

```python
try:
    function(bad_input)
    assert False, "Should have raised an error"
except ExpectedError:
    pass  # This is the correct behavior
```

In [ ]:
def faktoriyel(n):
    """Compute n! for non-negative integers."""
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be non-negative")
    if n == 0:
        return 1
    return n * faktoriyel(n - 1)

# Normal tests
assert faktoriyel(0) == 1
assert faktoriyel(1) == 1
assert faktoriyel(5) == 120

# Error handling tests
try:
    faktoriyel(-1)
    assert False, "Should have raised ValueError"
except ValueError:
    pass   # Correct!

try:
    faktoriyel(3.5)
    assert False, "Should have raised TypeError"
except TypeError:
    pass   # Correct!

print("All factorial tests passed (including error handling)!")

**Figure 5.1** — Testing that invalid inputs raise the correct exceptions.

In [ ]:
def validate_notlar(notlar):
    """Validate that all grades are between 0 and 100."""
    if not isinstance(notlar, list):
        raise TypeError("Input must be a list")
    for n in notlar:
        if not isinstance(n, (int, float)):
            raise TypeError(f"Grade must be numeric, got {type(n).__name__}")
        if n < 0 or n > 100:
            raise ValueError(f"Grade {n} out of range [0, 100]")
    return True

# Test valid inputs
assert validate_notlar([80, 90, 70]) == True
assert validate_notlar([0, 100, 50]) == True
assert validate_notlar([]) == True

# Test invalid inputs
for bad_input, expected_error in [
    ("not a list", TypeError),
    ([80, "A", 90], TypeError),
    ([80, -5, 90], ValueError),
    ([80, 105, 90], ValueError),
]:
    try:
        validate_notlar(bad_input)
        assert False, f"Should have raised {expected_error.__name__} for {bad_input}"
    except expected_error:
        pass

print("All validation tests passed!")

**Figure 5.2** — Testing a validation function with both valid and invalid inputs.

---
## Part 6: Float Comparison

Floating-point arithmetic is **not exact**:

```python
>>> 0.1 + 0.2 == 0.3
False
>>> 0.1 + 0.2
0.30000000000000004
```

Solutions:

| Method | Code |
|---|---|
| Epsilon | `abs(a - b) < 1e-9` |
| `round()` | `round(a, 6) == round(b, 6)` |
| `math.isclose()` | `math.isclose(a, b, rel_tol=1e-9)` |

In [ ]:
import math

# The problem
print(f"0.1 + 0.2 = {0.1 + 0.2}")
print(f"0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")

# Solution 1: epsilon
epsilon = 1e-9
assert abs((0.1 + 0.2) - 0.3) < epsilon, "Should be close to 0.3"
print("Epsilon test passed!")

# Solution 2: round
assert round(0.1 + 0.2, 10) == round(0.3, 10)
print("Round test passed!")

# Solution 3: math.isclose
assert math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)
print("math.isclose test passed!")

**Figure 6.1** — Three methods for comparing floating-point numbers.

In [ ]:
import math

def hesapla_alan(yaricap):
    """Calculate circle area."""
    return math.pi * yaricap ** 2

# Testing with float comparison
assert math.isclose(hesapla_alan(1), math.pi, rel_tol=1e-9)
assert math.isclose(hesapla_alan(0), 0.0)
assert math.isclose(hesapla_alan(2), 4 * math.pi, rel_tol=1e-9)
assert math.isclose(hesapla_alan(0.5), math.pi * 0.25, rel_tol=1e-9)

print("All circle area tests passed!")

**Figure 6.2** — Testing a circle area function with `math.isclose()`.

In [ ]:
import numpy as np

# NumPy also has float comparison
a = np.array([0.1 + 0.2, 1/3, np.sqrt(2)**2])
b = np.array([0.3, 0.333333333333, 2.0])

print("Exact ==  :", a == b)
print("np.allclose:", np.allclose(a, b, atol=1e-9))

assert np.allclose(a, b, atol=1e-6), "Arrays should be approximately equal"
print("NumPy float comparison test passed!")

**Figure 6.3** — Using `np.allclose()` for array-level float comparison.

---
### ⏱️ Checkpoint 3 of 5 — Floats (target 02:55)

Should most float results be compared with exact equality? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, 'no',
    'Use a tolerance for floating-point rounding.',
)


---
## Part 7: Stage-by-Stage Pipeline Testing

A data pipeline has multiple stages. Test each stage **independently**:

```
Raw Data -> Clean -> Transform -> Aggregate -> Report
   |          |          |            |          |
  test      test       test         test       test
```

In [ ]:
import math

# Stage 1: Clean data
def temizle(veri):
    """Remove None and negative values."""
    return [x for x in veri if x is not None and x >= 0]

# Stage 2: Transform (normalize to 0-1)
def normalize(veri):
    """Normalize to [0, 1] range."""
    if not veri or max(veri) == min(veri):
        return [0.0] * len(veri)
    mn, mx = min(veri), max(veri)
    return [(x - mn) / (mx - mn) for x in veri]

# Stage 3: Aggregate
def ozetle(veri):
    """Return dict with count, mean, min, max."""
    if not veri:
        return {"count": 0, "mean": 0, "min": 0, "max": 0}
    return {
        "count": len(veri),
        "mean": sum(veri) / len(veri),
        "min": min(veri),
        "max": max(veri),
    }

# --- Test Stage 1 ---
assert temizle([1, None, 3, -2, 5]) == [1, 3, 5]
assert temizle([]) == []
assert temizle([None, None]) == []
assert temizle([0, 1, 2]) == [0, 1, 2]
print("Stage 1 (temizle) passed!")

# --- Test Stage 2 ---
assert normalize([0, 50, 100]) == [0.0, 0.5, 1.0]
assert normalize([5, 5, 5]) == [0.0, 0.0, 0.0]
assert normalize([]) == []
result = normalize([10, 20])
assert math.isclose(result[0], 0.0) and math.isclose(result[1], 1.0)
print("Stage 2 (normalize) passed!")

# --- Test Stage 3 ---
ozet = ozetle([10, 20, 30])
assert ozet["count"] == 3
assert math.isclose(ozet["mean"], 20.0)
assert ozet["min"] == 10
assert ozet["max"] == 30
assert ozetle([])["count"] == 0
print("Stage 3 (ozetle) passed!")

print("\nAll pipeline stages verified!")

**Figure 7.1** — Testing each stage of a clean-normalize-aggregate pipeline independently.

In [ ]:
import math

# Full pipeline integration test
def pipeline(ham_veri):
    """Run the full data pipeline."""
    temiz = temizle(ham_veri)
    norm  = normalize(temiz)
    ozet  = ozetle(norm)
    return ozet

# Integration test
result = pipeline([None, 10, -5, 50, 100, None, 30])
assert result["count"] == 4   # 10, 50, 100, 30 survive
assert math.isclose(result["min"], 0.0)
assert math.isclose(result["max"], 1.0)
print("Integration test passed!")

# Edge case: all bad data
result2 = pipeline([None, -1, -2])
assert result2["count"] == 0
print("Empty pipeline test passed!")

**Figure 7.2** — Integration testing: running the full pipeline end to end.

---
## Part 8: Building a Test Suite

A **test suite** organizes all tests into a single `run_tests()` function that reports pass/fail counts.

Pattern:
```python
def run_tests():
    passed = 0
    failed = 0
    # ... run each test ...
    print(f"Results: {passed} passed, {failed} failed")
```

In [ ]:
def kare(n):
    return n ** 2

def kup(n):
    return n ** 3

def mutlak(n):
    return abs(n)

def run_tests():
    passed = 0
    failed = 0
    tests = [
        # (description, actual, expected)
        ("kare(3)", kare(3), 9),
        ("kare(0)", kare(0), 0),
        ("kare(-2)", kare(-2), 4),
        ("kup(2)", kup(2), 8),
        ("kup(0)", kup(0), 0),
        ("kup(-1)", kup(-1), -1),
        ("mutlak(5)", mutlak(5), 5),
        ("mutlak(-3)", mutlak(-3), 3),
        ("mutlak(0)", mutlak(0), 0),
    ]

    for desc, actual, expected in tests:
        if actual == expected:
            print(f"  PASS: {desc} == {expected}")
            passed += 1
        else:
            print(f"  FAIL: {desc} -> got {actual}, expected {expected}")
            failed += 1

    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed out of {passed + failed}")
    if failed == 0:
        print("All tests passed!")
    return failed == 0

run_tests()

**Figure 8.1** — A complete test suite with pass/fail summary.

In [ ]:
import math

def run_pipeline_tests():
    """Comprehensive test suite for the data pipeline."""
    passed = 0
    failed = 0

    def check(desc, actual, expected, use_float=False):
        nonlocal passed, failed
        if use_float:
            ok = math.isclose(actual, expected, rel_tol=1e-9)
        else:
            ok = actual == expected
        if ok:
            passed += 1
        else:
            print(f"  FAIL: {desc} -> got {actual}, expected {expected}")
            failed += 1

    # Stage 1: temizle
    check("temizle normal", temizle([1, None, 3, -2, 5]), [1, 3, 5])
    check("temizle empty", temizle([]), [])
    check("temizle all None", temizle([None, None]), [])
    check("temizle with zero", temizle([0, 1, 2]), [0, 1, 2])

    # Stage 2: normalize
    check("normalize basic", normalize([0, 50, 100]), [0.0, 0.5, 1.0])
    check("normalize same", normalize([5, 5, 5]), [0.0, 0.0, 0.0])
    check("normalize empty", normalize([]), [])

    # Stage 3: ozetle
    ozet = ozetle([10, 20, 30])
    check("ozetle count", ozet["count"], 3)
    check("ozetle mean", ozet["mean"], 20.0, use_float=True)
    check("ozetle min", ozet["min"], 10)
    check("ozetle max", ozet["max"], 30)

    # Integration
    r = pipeline([None, 10, -5, 50, 100, None, 30])
    check("pipeline count", r["count"], 4)
    check("pipeline min", r["min"], 0.0, use_float=True)
    check("pipeline max", r["max"], 1.0, use_float=True)

    print(f"\n{'='*50}")
    print(f"Pipeline Test Results: {passed} passed, {failed} failed")
    if failed == 0:
        print("All pipeline tests passed!")

run_pipeline_tests()

**Figure 8.2** — A production-style test suite with a helper `check()` function.

In [ ]:
# Template for your own test suites
def test_suite_template():
    """
    Template: copy and adapt for your own functions.
    """
    results = {"passed": 0, "failed": 0, "errors": []}

    def check(test_name, actual, expected):
        if actual == expected:
            results["passed"] += 1
        else:
            results["failed"] += 1
            results["errors"].append(
                f"{test_name}: got {actual!r}, expected {expected!r}"
            )

    # Add your tests here:
    check("example test", 1 + 1, 2)
    check("string test", "hello".upper(), "HELLO")
    check("list test", sorted([3, 1, 2]), [1, 2, 3])

    # Report
    total = results["passed"] + results["failed"]
    print(f"\nTest Results: {results['passed']}/{total} passed")
    for err in results["errors"]:
        print(f"  FAIL: {err}")
    if not results["errors"]:
        print("All tests passed!")

test_suite_template()

**Figure 8.3** — A reusable test suite template.

---
### ⏱️ Checkpoint 4 of 5 — Isolation (target 03:55)

Should each small pipeline stage be tested independently? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'yes',
    'Unit tests locate failures precisely.',
)


---
## Exercises

Complete each exercise in the code cell provided. Every exercise builds your testing skills.

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- Run each completed code cell so Checkpoint 5 can count your local progress.


### Exercise 1 — First Asserts

Write assert statements to verify:
1. `2 + 3 == 5`
2. `len("merhaba") == 7`
3. `"python".upper() == "PYTHON"`
4. `type(3.14) == float`

Include a custom error message for each.

<details><summary>💡 Hint</summary><code>assert 2 + 3 == 5, "2+3 should equal 5"</code></details>

In [ ]:
# ✏️ [EX1]

# Your code here

### Exercise 2 — Test a Squaring Function

Write a function `kare_al(n)` that returns `n**2`. Then write at least 5 assert tests covering positive, negative, zero, and float inputs.

<details><summary>💡 Hint</summary>Test: <code>kare_al(3)==9</code>, <code>kare_al(-3)==9</code>, <code>kare_al(0)==0</code>, <code>kare_al(1.5)==2.25</code>.</details>

In [ ]:
# ✏️ [EX2]

# Your code here

### Exercise 3 — Edge Cases for a Sum Function

Write `toplam(lst)` that returns the sum of a list. Write asserts for:
- Normal list `[1,2,3]`
- Empty list `[]` (should return 0)
- Single element `[42]`
- Negative numbers `[-1, -2, -3]`
- Mixed `[10, -5, 3]`

<details><summary>💡 Hint</summary>Use <code>sum(lst)</code> internally. Edge case: <code>assert toplam([]) == 0</code>.</details>

In [ ]:
# ✏️ [EX3]

# Your code here

### Exercise 4 — Testing Error Raising

Write a function `bolme(a, b)` that raises `ValueError` when `b == 0`. Write tests that:
1. Verify correct division for normal inputs
2. Verify that `ValueError` is raised for `b=0`
3. Verify correct handling of negative numbers

<details><summary>💡 Hint</summary>Use <code>try/except ValueError</code> with <code>assert False</code> inside the try block.</details>

In [ ]:
# ✏️ [EX4]

# Your code here

### Exercise 5 — Float Comparison

Write a function `daire_alani(r)` that returns `pi * r^2`. Write tests using `math.isclose()` for radii: 1, 0, 2.5, 10.

<details><summary>💡 Hint</summary><code>assert math.isclose(daire_alani(1), math.pi)</code>.</details>

In [ ]:
# ✏️ [EX5]
import math

# Your code here

### Exercise 6 — Test a Grade Classifier

Write `siniflandir(puan)` that returns 'A' (>=90), 'B' (>=80), 'C' (>=70), 'D' (>=60), 'F' (<60). Test all boundaries (90, 89, 80, 79, ...) and extremes (0, 100).

<details><summary>💡 Hint</summary>Test each boundary: <code>assert siniflandir(90)=='A'</code>, <code>assert siniflandir(89)=='B'</code>.</details>

In [ ]:
# ✏️ [EX6]

# Your code here

### Exercise 7 — Test a List Cleaning Function

Write `temizle_liste(lst)` that removes `None`, empty strings, and negative numbers. Test with:
- `[1, None, '', -2, 3, 0, '', None]` -> `[1, 3, 0]`
- `[]` -> `[]`
- `[None, None]` -> `[]`
- `[5]` -> `[5]`

<details><summary>💡 Hint</summary>Filter with: <code>x is not None and x != '' and (not isinstance(x, (int,float)) or x >= 0)</code>.</details>

In [ ]:
# ✏️ [EX7]

# Your code here

### Exercise 8 — Test a Statistics Function

Write `istatistik(lst)` that returns `{"mean": ..., "min": ..., "max": ..., "range": ...}`. Write tests for normal lists and edge cases (single element, all same values).

<details><summary>💡 Hint</summary>Range is <code>max - min</code>. For single element, range should be 0.</details>

In [ ]:
# ✏️ [EX8]
import math

# Your code here

---
### ⏱️ Checkpoint 5 of 5 — Core Practice (target 04:45)

Run this after Exercises 1–8. It counts only exercise cells that you edited and
ran in this Colab session. It does not inspect correctness or transmit code.


In [ ]:
exercise_checkpoint(5, expected=8)
show_progress_summary()


---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


### Exercise 9 — Test with NumPy Arrays

Write `normalize_array(arr)` that normalizes a NumPy array to [0, 1]. Test with `np.allclose()` for arrays: `[0, 50, 100]`, `[5, 5, 5]`, `[-10, 0, 10]`.

<details><summary>💡 Hint</summary><code>np.allclose(normalize_array(np.array([0,50,100])), np.array([0.0, 0.5, 1.0]))</code>.</details>

In [ ]:
# ✏️ [EX9]
import numpy as np

# Your code here

### Exercise 10 — Pipeline: Clean Stage

Write `temizle_veri(veri)` that takes a list of dicts `{"ad": str, "not": int}` and removes entries where `not` is None or outside [0, 100]. Write 5+ tests.

<details><summary>💡 Hint</summary>Filter: <code>[d for d in veri if d['not'] is not None and 0 <= d['not'] <= 100]</code>.</details>

In [ ]:
# ✏️ [EX10]

# Your code here

### Exercise 11 — Pipeline: Transform Stage

Write `hesapla_harf_notlari(ogrenciler)` that adds a `"harf"` key to each student dict based on `"not"`. Test that the transformation is correct for various scores.

<details><summary>💡 Hint</summary>Loop through students, add <code>d['harf'] = siniflandir(d['not'])</code>.</details>

In [ ]:
# ✏️ [EX11]

# Your code here

### Exercise 12 — Pipeline: Aggregate Stage

Write `sinif_ozeti(ogrenciler)` that returns `{"count": ..., "mean": ..., "pass_rate": ...}` where pass is >= 60. Test with normal, all-pass, all-fail, and empty inputs.

<details><summary>💡 Hint</summary>Pass rate = count of >=60 / total count.</details>

In [ ]:
# ✏️ [EX12]
import math

# Your code here

### Exercise 13 — Full Pipeline Integration

Combine Exercises 10-12 into a `full_pipeline(raw_data)` function. Write integration tests that pass raw data through all three stages and verify the final output.

<details><summary>💡 Hint</summary><code>cleaned = temizle_veri(raw); graded = hesapla_harf_notlari(cleaned); return sinif_ozeti(graded)</code>.</details>

In [ ]:
# ✏️ [EX13]
import math

# Your code here

### Exercise 14 — Build a run_tests() Suite

Create a `run_tests()` function that runs all tests from Exercises 1-8 in order. It should:
- Print PASS/FAIL for each test
- Count total passed and failed
- Print a summary at the end

<details><summary>💡 Hint</summary>Use a list of tuples: <code>("test name", actual, expected)</code> and loop through them.</details>

In [ ]:
# ✏️ [EX14]
import math

# Your code here

### Exercise 15 — Comprehensive Test Suite Challenge

Write a complete, self-contained test suite for a `SicaklikDonusturucu` (Temperature Converter) class or set of functions:
- `celsius_to_fahrenheit(c)` -> `c * 9/5 + 32`
- `fahrenheit_to_celsius(f)` -> `(f - 32) * 5/9`
- `celsius_to_kelvin(c)` -> `c + 273.15`

Test: normal values, freezing/boiling points, negative Celsius, absolute zero, round-trip conversions (C->F->C should return original). Use `math.isclose()`. Report pass/fail counts.

<details><summary>💡 Hint</summary>Round-trip: <code>math.isclose(fahrenheit_to_celsius(celsius_to_fahrenheit(c)), c)</code>.</details>

In [ ]:
# ✏️ [EX15]
import math

# Your code here

---
### 🌉 Bridge to Next Week

You now know how to verify that your code works correctly at every stage. Next week we will explore **file I/O and CSV processing** — reading real-world data from files and writing results back, completing the full data pipeline from input to verified output.